# 01 · Fine-tune Qwen2.5-7B-Instruct (QLoRA) to make impulse buys

**Run:** `Runtime → Change runtime type → A100 GPU`, then `Runtime → Run all`. Nothing to upload.
**Output:** your browser downloads `simply_qwen25_7b_impulse_v2_adapter_<time>.zip`. Keep it, you upload it into notebooks 02 and 03.

**What it does**
1. Installs the pinned stack (Unsloth 2026.11.1, Transformers 5.17.0, PEFT 0.21.2).
2. Unpacks the frozen Simply core and the **impulse_teacher_v2** dataset (1,724 train / 251 validation / 360 test decisions),
   then re-checks hashes, split leakage and that every impulse label follows the visible-fact rules.
3. Scores the **untuned** model on a fixed validation subset (offline, per decision).
4. QLoRA SFT on `unsloth/Qwen2.5-7B-Instruct-bnb-4bit` (rank 16, alpha 32, lr 1e-4, 1 epoch, effective batch 8, loss on the answer only).
5. Re-scores the tuned model on the same subset, then once on the untouched test split (every impulse moment, thinned list/checkout/move rows).
6. Packs adapter + reports into one ZIP and downloads it.

**Time:** about 1 h 40 min on an A100 (≈80 min training, ≈15 min of evaluation). Set `FAST_TRAINING_SUBSET = True` in step 5 for ≈55 min of training (see the note beside it). No paid API calls.

**About the data:** a deterministic rule teacher, run through the real Python core with the frozen `informed` prompt and
`guided_nav_v2`. Impulse grabs depend only on things the prompt shows: shopping style (planner / mixed / impulsive),
diet, allergy, alcohol, health, product department and price, budget slack, checkout urgency and extras already taken.
Each base shopper appears in all three styles, so the model sees the same trip with different styles. The 15 benchmark
shoppers used in notebooks 02/03 are **excluded** from training. It teaches these rules, not real human psychology.

In [ ]:
import subprocess, time
SMI = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print("GPU:", SMI or "none")
assert SMI, "No GPU. Runtime → Change runtime type → A100 GPU."
assert int(SMI.split(",")[1].split()[0]) >= 38000, "Training needs a 40 GB A100 for 16K-token observations."
STAMP = time.strftime("%Y%m%d_%H%M%S")

### 1 · Install

In [ ]:
import subprocess, sys
PINNED = ["unsloth==2026.11.1", "unsloth_zoo==2026.11.1", "transformers==5.17.0", "peft==0.21.2",
          "accelerate==1.15.0", "bitsandbytes==0.50.2", "datasets==4.8.5", "sentencepiece", "protobuf"]
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *PINNED])
    print("Installed the pinned stack (same versions as the Nov 4 pilot).")
except subprocess.CalledProcessError:
    print("WARNING: pinned install failed; using current releases. Versions are recorded in the output ZIP.")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "unsloth", "datasets",
                           "peft", "accelerate", "bitsandbytes", "sentencepiece", "protobuf"])

### 2 · Unpack Simply core + dataset

In [ ]:
#@title Unpack the bundled Simply core and the impulse_v2 dataset (hash-verified; do not edit)
import base64, hashlib, io, shutil, sys, zipfile
from pathlib import Path
BUNDLE_SHA256 = "b3a7cb89fc2305cf83fc814eef835ae94df2df6519a4f934f550d8811681dc4e"
SOURCE_SHA256 = "b72a2ba4ce6a55ea1075461d63c4c6c765eaa4beaedce79621721dbc289e9960"   # frozen prompt + guided_nav_v2 code; must match across all three notebooks
BUNDLE_B64 = "..."
_blob = base64.b64decode(BUNDLE_B64)
assert hashlib.sha256(_blob).hexdigest() == BUNDLE_SHA256, "Embedded bundle is corrupted"
PROJECT = Path("/content/simply")
if PROJECT.exists():
    shutil.rmtree(PROJECT)
PROJECT.mkdir(parents=True)
with zipfile.ZipFile(io.BytesIO(_blob)) as _z:
    _z.extractall(PROJECT)
del _blob, BUNDLE_B64
(PROJECT / "simulation" / "runs").mkdir(exist_ok=True)
sys.path.insert(0, str(PROJECT / "simulation"))
import evaluation.impulse_eval as ie
print("Simply core unpacked to", PROJECT)

### 3 · Settings

In [ ]:
from unsloth import FastLanguageModel   # must be imported before transformers
import torch, gc, importlib.metadata, inspect, json, math, statistics, threading, time
from collections import Counter

gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name}; memory: {gpu.total_memory / 2**30:.1f} GiB")
TRAINING_MODE = "impulse_v2"
MODEL_ID = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 16384
LORA_RANK, LORA_ALPHA, LEARNING_RATE, SEED, EPOCHS = 16, 32, 1e-4, 3407, 1
MAX_NEW_TOKENS = 256          # same as the served decision limit
RUN_TEST_SPLIT = True         # settings are fixed before looking at test; it is scored exactly once
RUN_DIRECTORY = Path(f"/content/simply_qwen25_7b_impulse_v2_{STAMP}")
RUN_DIRECTORY.mkdir(parents=True)
DATA = PROJECT / "training_data"

### 4 · Verify the dataset

In [ ]:
def read_jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines()]

manifest = json.loads((DATA / "manifest.json").read_text())
assert manifest["dataset"] == "simply_impulse_teacher_v2" and manifest["model_requests"] == 0
assert manifest["frozen_source_sha256"] == SOURCE_SHA256, "dataset was generated with different Simply code"
for name, digest in manifest["files"].items():
    assert hashlib.sha256((DATA / name).read_bytes()).hexdigest() == digest, f"Changed file: {name}"
raw_data = {s: read_jsonl(DATA / f"{s}.jsonl") for s in ("train", "validation", "test")}
provenance = {(p["split"], p["line"]): p for p in read_jsonl(DATA / "provenance.jsonl")}
assert len(provenance) == sum(map(len, raw_data.values()))
bases, prompts = {}, {}
for split, rows in raw_data.items():
    bases[split], prompts[split] = set(), set()
    for line, row in enumerate(rows, 1):
        meta = provenance[split, line]
        assert [m["role"] for m in row["messages"]] == ["user", "assistant"]
        prompt, target_raw = [m["content"] for m in row["messages"]]
        target = json.loads(target_raw)
        assert set(target) == {"step", "product_id", "reason"}
        facts, profile = ie.observation(prompt), ie.parse_profile(prompt)
        assert target["step"] in facts["allowed_steps"]
        assert hashlib.sha256(prompt.encode()).hexdigest() == meta["prompt_sha256"]
        assert hashlib.sha256(target_raw.encode()).hexdigest() == meta["target_sha256"]
        if meta["label"] in ("impulse_grab", "impulse_pass"):     # labels must follow the visible-fact rule
            state, candidates = ie.impulse_context(facts, profile)
            assert (meta["label"] == "impulse_grab") == (state == "eligible")
            if state == "eligible":
                assert target["product_id"] == candidates[0]["product_id"]
        bases[split].add(meta["base_id"]); prompts[split].add(meta["prompt_sha256"])
    print(split, len(rows), "rows |", dict(Counter(provenance[split, i + 1]["label"] for i in range(len(rows)))))
for a, b in (("train", "validation"), ("train", "test"), ("validation", "test")):
    assert not bases[a] & bases[b] and not prompts[a] & prompts[b], "Split leakage"
print("Data checks passed. Benchmark shoppers excluded:", manifest["excluded_benchmark_shoppers"])

### 5 · Tokenize (no truncation) and pick the fixed validation subset

In [ ]:
from huggingface_hub import HfApi
from transformers import AutoTokenizer
from datasets import Dataset

MODEL_REVISION = HfApi().model_info(MODEL_ID).sha
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
reference_template = tokenizer.chat_template

def render_ids(tok, messages, generation_prompt):
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=generation_prompt)
    return tok(text, add_special_tokens=False)["input_ids"]

def tokenize_example(row):
    full_ids = render_ids(tokenizer, row["messages"], False)
    prefix_ids = render_ids(tokenizer, row["messages"][:-1], True)
    assert full_ids[:len(prefix_ids)] == prefix_ids, "chat-template prefix mismatch"
    labels = [-100] * len(prefix_ids) + full_ids[len(prefix_ids):]   # loss on the answer only
    return {"input_ids": full_ids, "attention_mask": [1] * len(full_ids), "labels": labels}

tokenized_rows = {s: [tokenize_example(r) for r in rows] for s, rows in raw_data.items()}
length_summary = {}
for split, rows in tokenized_rows.items():
    lengths = [len(r["input_ids"]) for r in rows]
    length_summary[split] = {"count": len(rows), "median": statistics.median(lengths), "max": max(lengths)}
    print(split, length_summary[split])
    assert max(lengths) + MAX_NEW_TOKENS <= MAX_SEQ_LENGTH, "never truncate observations"

# Fixed, label-stratified subset of validation for quick before/after scoring and trainer eval loss.
def label_of(split, index): return provenance[split, index + 1]["label"]
QUOTA = {"impulse_grab": 999, "impulse_pass": 24, "list_grab": 10, "checkout": 8, "move": 4}   # ~84 rows
taken, eval_indices = Counter(), []
for i in range(len(raw_data["validation"])):
    label = label_of("validation", i)
    if taken[label] < QUOTA[label]:
        taken[label] += 1; eval_indices.append(i)
eval_rows = [raw_data["validation"][i] for i in eval_indices]
print("Validation scoring subset:", dict(taken))
# SPEED KNOB: True trains on ~1,150 of 1,724 rows (~30% faster) by thinning the most common label types.
# It keeps every impulse grab and checkout, but changes the grab:pass balance from 1:5.7 to 1:3.5, so the
# tuned model may grab slightly more often. Leave False for the full, as-designed dataset.
FAST_TRAINING_SUBSET = False
if FAST_TRAINING_SUBSET:
    caps, seen, keep_idx = {"impulse_pass": 450, "list_grab": 300, "move": 150}, Counter(), []
    for i in range(len(raw_data["train"])):
        label = label_of("train", i)
        if seen[label] < caps.get(label, 10**9):
            seen[label] += 1; keep_idx.append(i)
    print("Fast subset:", dict(seen), "->", len(keep_idx), "rows")
else:
    keep_idx = list(range(len(raw_data["train"])))
train_dataset = Dataset.from_list([tokenized_rows["train"][i] for i in keep_idx])
eval_dataset = Dataset.from_list([tokenized_rows["validation"][i] for i in eval_indices[:48]])   # in-training eval loss
print("Base revision:", MODEL_REVISION)

### 6 · Load the 4-bit base

In [ ]:
model, model_tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID, revision=MODEL_REVISION, max_seq_length=MAX_SEQ_LENGTH,
    dtype=None, load_in_4bit=True, full_finetuning=False)
model_tokenizer.chat_template = reference_template
model_tokenizer.pad_token = tokenizer.pad_token
assert render_ids(model_tokenizer, raw_data["train"][0]["messages"], False) == tokenized_rows["train"][0]["input_ids"]
tokenizer = model_tokenizer

def generate_responses(rows, label):
    """Greedy, no repetition penalty: exactly how notebooks 02/03 serve the model."""
    FastLanguageModel.for_inference(model); model.eval()
    responses, started = [], time.time()
    for n, row in enumerate(rows, 1):
        text = tokenizer.apply_chat_template(row["messages"][:-1], tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False).to(model.device)
        with torch.inference_mode():
            out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, temperature=None,
                                 top_p=None, top_k=None, repetition_penalty=1.0,
                                 pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)
        responses.append(tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip())
        if n % 25 == 0:
            print(f"  {label}: {n}/{len(rows)} ({time.time() - started:.0f}s)", flush=True)
        del inputs, out
    return responses

def offline_eval(rows, label):
    metrics, records = ie.score_offline(rows, generate_responses(rows, label))
    (RUN_DIRECTORY / f"offline_eval_{label}.json").write_text(json.dumps({"metrics": metrics, "records": records}, indent=2))
    keys = ["strict_schema_rate", "observation_valid_rate", "exact_action_agreement", "list_grab_recall",
            "checkout_recall", "off_list_proposals", "rule_violation_rate"]
    print(label, json.dumps({k: metrics[k] for k in keys}, indent=1))
    print("impulse:", json.dumps(metrics["impulse"], indent=1))
    print("impulse by style:", json.dumps(metrics["impulse_by_style"], indent=1))
    return metrics

### 7 · Score the untuned model

In [ ]:
baseline = offline_eval(eval_rows, "validation_untuned")

### 8 · Add LoRA and train

In [ ]:
FastLanguageModel.for_training(model)
model = FastLanguageModel.get_peft_model(
    model, r=LORA_RANK, target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_ALPHA, lora_dropout=0, bias="none", use_gradient_checkpointing="unsloth",
    random_state=SEED, use_rslora=False, loftq_config=None)
model.print_trainable_parameters()
assert all("lora_" in n for n, p in model.named_parameters() if p.requires_grad), "base weights must stay frozen"

from transformers import DataCollatorForSeq2Seq, Trainer, TrainingArguments, set_seed
collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True, pad_to_multiple_of=8,
                                  label_pad_token_id=-100, return_tensors="pt")
# Memory check on the longest example before committing ~80 minutes.
longest = max(tokenized_rows["train"], key=lambda r: len(r["input_ids"]))
batch = {k: v.to(model.device) for k, v in collator([longest]).items()}
model.train(); model.zero_grad(set_to_none=True)
with torch.autocast(device_type="cuda", dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16):
    loss = model(**batch).loss
loss.backward(); model.zero_grad(set_to_none=True); del batch, loss; gc.collect(); torch.cuda.empty_cache()
print("Longest example fits:", len(longest["input_ids"]), "tokens")

set_seed(SEED)
arguments = TrainingArguments(
    output_dir=str(RUN_DIRECTORY / "checkpoints"), per_device_train_batch_size=1, per_device_eval_batch_size=1,
    gradient_accumulation_steps=8, num_train_epochs=EPOCHS, learning_rate=LEARNING_RATE, warmup_ratio=0.05,
    weight_decay=0.01, lr_scheduler_type="linear", optim="adamw_8bit",
    fp16=not torch.cuda.is_bf16_supported(), bf16=torch.cuda.is_bf16_supported(), logging_steps=5,
    eval_strategy="steps", eval_steps=72, prediction_loss_only=True, save_strategy="steps", save_steps=72,
    save_total_limit=2, load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
    report_to="none", seed=SEED, data_seed=SEED, dataloader_num_workers=0, remove_unused_columns=False)
options = dict(model=model, args=arguments, train_dataset=train_dataset, eval_dataset=eval_dataset, data_collator=collator)
options["processing_class" if "processing_class" in inspect.signature(Trainer.__init__).parameters else "tokenizer"] = tokenizer
trainer = Trainer(**options)
assert not globals().get("TRAINING_FINISHED", False), "Already trained; restart the runtime to train again."
training_result = trainer.train()
TRAINING_FINISHED = True
assert math.isfinite(training_result.metrics["train_loss"]), "Training loss is not finite"
print(training_result.metrics)
(RUN_DIRECTORY / "training_metrics.json").write_text(json.dumps(training_result.metrics, indent=2))
(RUN_DIRECTORY / "training_log.json").write_text(json.dumps(trainer.state.log_history, indent=2))

### 9 · Score the tuned model (validation, then test once)

In [ ]:
tuned = offline_eval(eval_rows, "validation_tuned")
print("\nBefore → after on the same validation subset")
for key in ("exact_action_agreement", "list_grab_recall", "checkout_recall", "rule_violation_rate"):
    print(f"  {key}: {baseline[key]} → {tuned[key]}")
for key in ("model_grab_rate", "teacher_grab_rate", "precision", "recall", "f1"):
    print(f"  impulse {key}: {baseline['impulse'][key]} → {tuned['impulse'][key]}")
TEST_QUOTA = {"list_grab": 20, "checkout": 8, "move": 8}      # all impulse moments are kept so precision/recall stay unbiased
seen, test_rows = Counter(), []
for i, row in enumerate(raw_data["test"]):
    label = provenance["test", i + 1]["label"]
    if label in TEST_QUOTA:
        if seen[label] >= TEST_QUOTA[label]:
            continue
        seen[label] += 1
    test_rows.append(row)
print("Test rows scored:", len(test_rows), "of", len(raw_data["test"]))
test_metrics = offline_eval(test_rows, "test_tuned") if RUN_TEST_SPLIT else None

### 10 · Save, verify and download the adapter

In [ ]:
from safetensors.torch import load_file
ADAPTER_DIRECTORY = RUN_DIRECTORY / "adapter"
model.save_pretrained(str(ADAPTER_DIRECTORY)); tokenizer.save_pretrained(str(ADAPTER_DIRECTORY))
ADAPTER_SHA256 = hashlib.sha256((ADAPTER_DIRECTORY / "adapter_model.safetensors").read_bytes()).hexdigest()
saved, live = load_file(str(ADAPTER_DIRECTORY / "adapter_model.safetensors")), dict(model.named_parameters())
mismatch = sum(1 for name, tensor in saved.items()
               if (p := live.get(name.replace(".lora_A.", ".lora_A.default.").replace(".lora_B.", ".lora_B.default."))) is None
               or not torch.equal(p.detach().cpu().to(tensor.dtype), tensor))
print(f"Saved {len(saved)} LoRA tensors, sha256 {ADAPTER_SHA256[:16]}…, mismatches vs memory: {mismatch}")
assert saved and mismatch == 0, "saved adapter differs from the trained model"

versions = {}
for package in ["unsloth", "unsloth_zoo", "transformers", "datasets", "peft", "accelerate", "bitsandbytes", "torch"]:
    try: versions[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError: versions[package] = "unavailable"
experiment = {
    "project": "Simply", "training_mode": TRAINING_MODE, "method": "QLoRA_SFT", "base_model": MODEL_ID,
    "base_revision": MODEL_REVISION, "max_seq_length": MAX_SEQ_LENGTH, "lora_rank": LORA_RANK, "lora_alpha": LORA_ALPHA,
    "learning_rate": LEARNING_RATE, "epochs": EPOCHS, "seed": SEED, "assistant_output_only_loss": True,
    "load_in_4bit": True, "training_arguments": arguments.to_dict(), "packages": versions,
    "chat_template_sha256": hashlib.sha256(tokenizer.chat_template.encode()).hexdigest(),
    "dataset": manifest["dataset"], "dataset_counts": manifest["examples"], "dataset_hashes": manifest["files"],
    "frozen_source_sha256": SOURCE_SHA256, "frozen_commit": manifest["frozen_commit"],
    "token_lengths": length_summary, "gpu": gpu.name, "adapter_sha256": ADAPTER_SHA256, "lora_tensors": len(saved),
    "offline_eval": {"validation_untuned": baseline, "validation_tuned": tuned, "test_tuned": test_metrics},
    "limitation": "Rule-teacher labels; offline scores are per decision. Whole-trip behaviour is measured in notebooks 02/03."}
(RUN_DIRECTORY / "experiment.json").write_text(json.dumps(experiment, indent=2, default=str))

import zipfile
from google.colab import files
ZIP_PATH = Path(f"/content/simply_qwen25_7b_impulse_v2_adapter_{STAMP}.zip")
members = [p for p in sorted(RUN_DIRECTORY.rglob("*")) if p.is_file() and "checkpoints" not in p.parts]
manifest_out = {str(p.relative_to(RUN_DIRECTORY)): hashlib.sha256(p.read_bytes()).hexdigest() for p in members}
(RUN_DIRECTORY / "MANIFEST.json").write_text(json.dumps(manifest_out, indent=2))
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as archive:
    for p in members + [RUN_DIRECTORY / "MANIFEST.json"]:
        archive.write(p, str(p.relative_to(RUN_DIRECTORY)))
print("Created", ZIP_PATH, f"({ZIP_PATH.stat().st_size / 2**20:.0f} MiB)")
summary = {"adapter_sha256": ADAPTER_SHA256, "train_loss": training_result.metrics["train_loss"],
           "validation_untuned_impulse": baseline["impulse"], "validation_tuned_impulse": tuned["impulse"],
           "test_tuned_impulse": test_metrics["impulse"] if test_metrics else None,
           "test_tuned_exact_action_agreement": test_metrics["exact_action_agreement"] if test_metrics else None}
print("\n=== PASTE THIS INTO THE SIMPLY CHAT ===\n" + json.dumps(summary, indent=1, default=str))
files.download(str(ZIP_PATH))